# Input pool 

Builds the input pool the search picks from, for both backbones (CLIP ViT-B/16 and EfficientNet-B4):

- **128 PCA components**, fitted on all of FF++ train (no labels).
- **16 PLS components**, fitted on search-train only, so the inner fitness frames never touch a label.
- Every column is mapped to an angle in [0, pi] using the 0.5 and 99.5 percentiles on train.

In [1]:
backbones = {"clip": "_clip.npy", "effb4": "_effb4.npy"}
names = ["ffpp_train", "ffpp_val", "ffpp_test", "celebdf_test"]
n_pca = 128
n_pls = 16
low, high = 0.5, 99.5
out_dir = "/kaggle/working/pool"
seed = 0

In [2]:
import os
import glob
import json

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

os.makedirs(out_dir, exist_ok=True)

## Load

Both backbones were extracted from the same manifests, so their rows must match line by line. The check below stops the notebook if they don't.

In [3]:
def find(pattern):
    for depth in range(8):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + pattern)
        if len(hits) > 0:
            return hits[0]
    raise FileNotFoundError(pattern)


feats = {b: {} for b in backbones}
rows = {}
for b, suffix in backbones.items():
    for name in names:
        path = find(name + suffix)
        feats[b][name] = np.load(path).astype(np.float64)
        r = pd.read_csv(os.path.join(os.path.dirname(path), name + "_rows.csv"), dtype={"video_id": str})
        assert len(r) == len(feats[b][name]), (b, name)
        if name in rows:
            assert (r["path"].values == rows[name]["path"].values).all(), ("rows differ", b, name)
        else:
            rows[name] = r
        print(b, name, feats[b][name].shape)

is_inner = np.load(find("ffpp_train_is_inner.npy")).astype(bool)
assert len(is_inner) == len(rows["ffpp_train"])
search = ~is_inner
y = {name: rows[name]["label"].to_numpy() for name in names}
print("search-train frames", search.sum(), "| inner frames", is_inner.sum())

clip ffpp_train (46076, 512)
clip ffpp_val (22396, 512)
clip ffpp_test (22400, 512)
clip celebdf_test (16572, 512)
effb4 ffpp_train (46076, 1792)
effb4 ffpp_val (22396, 1792)
effb4 ffpp_test (22400, 1792)
effb4 celebdf_test (16572, 1792)
search-train frames 39164 | inner frames 6912


## PCA, PLS and angles

Each feature vector is scaled to length 1 first, as in Phase 1. PCA sees all of train; PLS sees only search-train frames and their labels.

In [4]:
pool = {b: {} for b in backbones}
columns = [f"pca{j}" for j in range(n_pca)] + [f"pls{j}" for j in range(n_pls)]

for b in backbones:
    xn = {name: feats[b][name] / np.linalg.norm(feats[b][name], axis=1, keepdims=True) for name in names}

    pca = PCA(n_components=n_pca, svd_solver="full", random_state=seed)
    pca.fit(xn["ffpp_train"])
    pls = PLSRegression(n_components=n_pls, scale=True)
    pls.fit(xn["ffpp_train"][search], y["ffpp_train"][search])

    scores = {name: np.hstack([pca.transform(xn[name]), pls.transform(xn[name])]) for name in names}
    lo = np.percentile(scores["ffpp_train"], low, axis=0)
    hi = np.percentile(scores["ffpp_train"], high, axis=0)
    for name in names:
        pool[b][name] = np.clip((scores[name] - lo) / (hi - lo), 0.0, 1.0) * np.pi
        np.save(f"{out_dir}/{b}_{name}_pool.npy", pool[b][name].astype(np.float32))
        np.save(f"{out_dir}/{b}_{name}_scores.npy", scores[name].astype(np.float32))

    np.savez(f"{out_dir}/{b}_pool_params.npz", pca_components=pca.components_, pca_mean=pca.mean_,
             pca_variance_ratio=pca.explained_variance_ratio_, pls_rotations=pls.x_rotations_,
             pls_x_mean=pls._x_mean, pls_x_std=pls._x_std, angle_lo=lo, angle_hi=hi)
    cum = np.cumsum(pca.explained_variance_ratio_)
    print(b, "variance kept by 8, 32, 128 PCA components: %.1f%%, %.1f%%, %.1f%%"
          % (100 * cum[7], 100 * cum[31], 100 * cum[127]))

for name in names:
    rows[name].to_csv(f"{out_dir}/{name}_rows.csv", index=False)
np.save(f"{out_dir}/ffpp_train_is_inner.npy", is_inner)
with open(f"{out_dir}/columns.json", "w") as f:
    json.dump(columns, f)

clip variance kept by 8, 32, 128 PCA components: 42.5%, 69.4%, 92.6%
effb4 variance kept by 8, 32, 128 PCA components: 48.8%, 70.4%, 87.7%


## Per-column strength

AUC of each pool column on its own, on search-train and on val. A column can point either way, so strength is max(auc, 1 - auc). The greedy and mutual-information baselines will use the search-train numbers only.

In [5]:
strength = {}
for b in backbones:
    table = []
    for j, col in enumerate(columns):
        a_st = roc_auc_score(y["ffpp_train"][search], pool[b]["ffpp_train"][search, j])
        a_val = roc_auc_score(y["ffpp_val"], pool[b]["ffpp_val"][:, j])
        table.append({"column": col, "strength_search_train": max(a_st, 1 - a_st), "strength_val": max(a_val, 1 - a_val)})
    strength[b] = pd.DataFrame(table)
    strength[b].to_csv(f"{out_dir}/{b}_column_strength.csv", index=False)
    top = strength[b].sort_values("strength_search_train", ascending=False).head(8)
    print(b, "strongest 8 on search-train:", top["column"].tolist())

clip strongest 8 on search-train: ['pls0', 'pls1', 'pca8', 'pls3', 'pca17', 'pls2', 'pca3', 'pls5']
effb4 strongest 8 on search-train: ['pls4', 'pls6', 'pls7', 'pls0', 'pls1', 'pls2', 'pls3', 'pls5']


## Bottleneck table on val

Logistic regression trained on all of FF++ train, scored on val at video level (a video's score is the mean of its frame probabilities). One fixed setting (standardized, C = 1): a first look, not the tuned baselines of week 2.

Two kinds of rows:
- **raw**: the PCA or PLS scores themselves, as in the Phase 1 table.
- **twin**: [sin, cos] of the 8 angles. This is exactly the function class of the additive circuit (RY encoding, single-qubit rotations, weighted sum of Z), so it shows what that circuit can reach with those 8 inputs.

In [6]:
val_video = rows["ffpp_val"].groupby(["method", "video_id"]).ngroup().to_numpy()


def video_auc(prob):
    df = pd.DataFrame({"video": val_video, "p": prob, "y": y["ffpp_val"]})
    g = df.groupby("video").agg(p=("p", "mean"), y=("y", "first"))
    return roc_auc_score(g["y"], g["p"])


def twin(a):
    return np.hstack([np.sin(a), np.cos(a)])


def fit_score(xt, xv):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000))
    model.fit(xt, y["ffpp_train"])
    return video_auc(model.predict_proba(xv)[:, 1])


results = []
for b in backbones:
    s = {name: np.load(f"{out_dir}/{b}_{name}_scores.npy").astype(np.float64) for name in ["ffpp_train", "ffpp_val"]}
    a = {name: pool[b][name] for name in ["ffpp_train", "ffpp_val"]}
    best8 = strength[b].sort_values("strength_search_train", ascending=False).index[:8].tolist()
    pls8 = list(range(n_pca, n_pca + 8))
    first8 = list(range(8))
    xn_t = feats[b]["ffpp_train"] / np.linalg.norm(feats[b]["ffpp_train"], axis=1, keepdims=True)
    xn_v = feats[b]["ffpp_val"] / np.linalg.norm(feats[b]["ffpp_val"], axis=1, keepdims=True)
    inputs = [
        ("raw first 8 PCA", s["ffpp_train"][:, first8], s["ffpp_val"][:, first8]),
        ("raw best 8 of pool", s["ffpp_train"][:, best8], s["ffpp_val"][:, best8]),
        ("raw PLS 8", s["ffpp_train"][:, pls8], s["ffpp_val"][:, pls8]),
        ("raw PLS 16", s["ffpp_train"][:, n_pca:], s["ffpp_val"][:, n_pca:]),
        ("raw all 144", s["ffpp_train"], s["ffpp_val"]),
        ("twin first 8 PCA", twin(a["ffpp_train"][:, first8]), twin(a["ffpp_val"][:, first8])),
        ("twin best 8 of pool", twin(a["ffpp_train"][:, best8]), twin(a["ffpp_val"][:, best8])),
        ("twin PLS 8", twin(a["ffpp_train"][:, pls8]), twin(a["ffpp_val"][:, pls8])),
        ("full linear probe", xn_t, xn_v),
    ]
    for label, xt, xv in inputs:
        results.append({"backbone": b, "input": label, "video auc": round(fit_score(xt, xv), 3)})

results = pd.DataFrame(results)
results.to_csv(f"{out_dir}/bottleneck_table.csv", index=False)
print(results.pivot(index="input", columns="backbone", values="video auc").loc[[r[0] for r in inputs]].to_string())

backbone              clip  effb4
input                            
raw first 8 PCA      0.636  0.522
raw best 8 of pool   0.828  0.692
raw PLS 8            0.850  0.692
raw PLS 16           0.873  0.720
raw all 144          0.882  0.725
twin first 8 PCA     0.645  0.519
twin best 8 of pool  0.832  0.689
twin PLS 8           0.853  0.689
full linear probe    0.893  0.764


## Saved files

`{backbone}_{split}_pool.npy` (angles, 144 columns), `{backbone}_{split}_scores.npy` (the raw scores), `{backbone}_pool_params.npz`, `{backbone}_column_strength.csv`, `columns.json`, the rows files, `ffpp_train_is_inner.npy` and `bottleneck_table.csv`. Save this output as a dataset for the next notebooks.

In [7]:
!ls -lh {out_dir}

total 250M
-rw-r--r-- 1 root root  498 Nov 26 12:46 bottleneck_table.csv
-rw-r--r-- 1 root root 2.3M Nov 26 12:45 celebdf_test_rows.csv
-rw-r--r-- 1 root root 9.2M Nov 26 12:44 clip_celebdf_test_pool.npy
-rw-r--r-- 1 root root 9.2M Nov 26 12:44 clip_celebdf_test_scores.npy
-rw-r--r-- 1 root root 6.2K Nov 26 12:45 clip_column_strength.csv
-rw-r--r-- 1 root root  13M Nov 26 12:44 clip_ffpp_test_pool.npy
-rw-r--r-- 1 root root  13M Nov 26 12:44 clip_ffpp_test_scores.npy
-rw-r--r-- 1 root root  26M Nov 26 12:44 clip_ffpp_train_pool.npy
-rw-r--r-- 1 root root  26M Nov 26 12:44 clip_ffpp_train_scores.npy
-rw-r--r-- 1 root root  13M Nov 26 12:44 clip_ffpp_val_pool.npy
-rw-r--r-- 1 root root  13M Nov 26 12:44 clip_ffpp_val_scores.npy
-rw-r--r-- 1 root root 594K Nov 26 12:44 clip_pool_params.npz
-rw-r--r-- 1 root root 1.3K Nov 26 12:45 columns.json
-rw-r--r-- 1 root root 9.2M Nov 26 12:45 effb4_celebdf_test_pool.npy
-rw-r--r-- 1 root root 9.2M Nov 26 12:45 effb4_celebdf_test_scores.npy
-rw-r--r